# 04 - Plate OCR Dataset Labeling

Notebook ini mengubah anotasi polygon Test V2 menjadi crop plat yang diluruskan, menyediakan UI labeling teks, menyimpan progres secara otomatis, dan mengekspor manifest untuk fine-tuning recognizer. Split `test` hanya untuk evaluasi dan tidak boleh masuk training.

## 1. Setup

Pilih kernel `.venv`, lalu jalankan semua cell sampai UI labeling. Data hasil labeling disimpan di `data/processed/plate_ocr` yang tidak boleh di-commit.

In [ ]:
import re
from pathlib import Path

import cv2
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage, clear_output, display

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ml').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Project root tidak ditemukan.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_ROOT = PROJECT_ROOT / 'TEMPORARY/Test V2'
OUTPUT_ROOT = PROJECT_ROOT / 'data/processed/plate_ocr'
CROPS_ROOT = OUTPUT_ROOT / 'images'
ANNOTATION_PATH = OUTPUT_ROOT / 'annotations.csv'
ACTIVE_LEARNING_QUEUE_PATH = OUTPUT_ROOT / 'active_learning_queue.csv'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CROPS_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
PLATE_PATTERN = re.compile(r'^[A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3}$')
print(f'Source: {SOURCE_ROOT}')
print(f'Output: {OUTPUT_ROOT}')


## 2. Build annotation index and rectified crops

In [ ]:
def normalize_plate_text(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(value).upper())

def read_yolo_shapes(label_path: Path, image_width: int, image_height: int) -> list[np.ndarray]:
    shapes = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        values = np.asarray([float(value) for value in parts[1:]], dtype=np.float32)
        if len(values) == 4:
            cx, cy, width, height = values
            points = np.asarray([[cx - width / 2, cy - height / 2], [cx + width / 2, cy - height / 2], [cx + width / 2, cy + height / 2], [cx - width / 2, cy + height / 2]], dtype=np.float32)
        elif len(values) >= 8 and len(values) % 2 == 0:
            points = values.reshape(-1, 2)
            if len(points) > 4 and np.allclose(points[0], points[-1]):
                points = points[:-1]
        else:
            continue
        points[:, 0] *= image_width
        points[:, 1] *= image_height
        shapes.append(points)
    return shapes

def order_quad(points: np.ndarray) -> np.ndarray:
    sums = points.sum(axis=1)
    differences = np.diff(points, axis=1).ravel()
    return np.asarray([points[np.argmin(sums)], points[np.argmin(differences)], points[np.argmax(sums)], points[np.argmax(differences)]], dtype=np.float32)

def rectify_plate(image: np.ndarray, polygon: np.ndarray, padding: float = 0.08) -> np.ndarray | None:
    rectangle = cv2.boxPoints(cv2.minAreaRect(polygon.astype(np.float32)))
    source = order_quad(rectangle)
    center = source.mean(axis=0)
    source = center + (source - center) * (1.0 + 2.0 * padding)
    source[:, 0] = np.clip(source[:, 0], 0, image.shape[1] - 1)
    source[:, 1] = np.clip(source[:, 1], 0, image.shape[0] - 1)
    top_width = np.linalg.norm(source[1] - source[0])
    bottom_width = np.linalg.norm(source[2] - source[3])
    left_height = np.linalg.norm(source[3] - source[0])
    right_height = np.linalg.norm(source[2] - source[1])
    target_width = max(1, int(max(top_width, bottom_width)))
    target_height = max(1, int(max(left_height, right_height)))
    if target_width < 8 or target_height < 4:
        return None
    destination = np.asarray([[0, 0], [target_width - 1, 0], [target_width - 1, target_height - 1], [0, target_height - 1]], dtype=np.float32)
    matrix = cv2.getPerspectiveTransform(source, destination)
    crop = cv2.warpPerspective(image, matrix, (target_width, target_height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    if crop.shape[0] > crop.shape[1]:
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
    return crop

def source_reference_id(image_path: Path) -> str:
    source_name = image_path.name.split('_jpg.rf.')[0]
    match = re.search(r'\d{8}_\d+_(.+)$', source_name)
    return normalize_plate_text(match.group(1) if match else '')


In [ ]:
REBUILD_CROPS = False  # Ubah True hanya jika algoritme rectification berubah.
SPLITS = ('train', 'valid', 'test')

existing_df = pd.read_csv(ANNOTATION_PATH).fillna('') if ANNOTATION_PATH.is_file() else pd.DataFrame()
existing_labels = existing_df.set_index('sample_id').to_dict('index') if not existing_df.empty else {}
index_rows = []

for split in SPLITS:
    images_dir = SOURCE_ROOT / split / 'images'
    labels_dir = SOURCE_ROOT / split / 'labels'
    split_crop_dir = CROPS_ROOT / split
    split_crop_dir.mkdir(parents=True, exist_ok=True)
    image_paths = sorted(path for path in images_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        label_path = labels_dir / f'{image_path.stem}.txt'
        if image is None or not label_path.is_file():
            continue
        polygons = read_yolo_shapes(label_path, image.shape[1], image.shape[0])
        for plate_index, polygon in enumerate(polygons, start=1):
            sample_id = f'{split}__{image_path.stem}__p{plate_index:02d}'
            crop_path = split_crop_dir / f'{sample_id}.jpg'
            if REBUILD_CROPS or not crop_path.is_file():
                crop = rectify_plate(image, polygon)
                if crop is None:
                    continue
                cv2.imwrite(str(crop_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            previous = existing_labels.get(sample_id, {})
            index_rows.append({'sample_id': sample_id, 'split': split, 'source_file': image_path.name, 'source_path': str(image_path), 'crop_path': str(crop_path), 'reference_id': source_reference_id(image_path), 'plate_text': previous.get('plate_text', ''), 'status': previous.get('status', 'PENDING'), 'notes': previous.get('notes', '')})

annotations_df = pd.DataFrame(index_rows)
annotations_df.to_csv(ANNOTATION_PATH, index=False)
display(annotations_df.groupby(['split', 'status']).size().rename('samples').reset_index())
print(f'Annotation index: {ANNOTATION_PATH}')


## 3. Manual labeling UI

Pilih split. Untuk benchmark, selesaikan `test` terlebih dahulu. Tombol Save menyimpan label lalu berpindah ke sampel berikutnya. `SKIPPED` digunakan jika plat tidak dapat dibaca secara objektif.

In [ ]:
split_selector = widgets.ToggleButtons(options=['test', 'valid', 'train'], value='test', description='Split')
show_selector = widgets.ToggleButtons(options=['Pending', 'All', 'Labeled', 'Skipped'], value='Pending', description='Show')
plate_input = widgets.Text(description='Plate', placeholder='B1234ABC', layout=widgets.Layout(width='420px'))
notes_input = widgets.Text(description='Notes', placeholder='blur, glare, unreadable, etc.', layout=widgets.Layout(width='620px'))
previous_button = widgets.Button(description='Previous', icon='arrow-left')
save_button = widgets.Button(description='Save & Next', icon='check', button_style='success')
skip_button = widgets.Button(description='Skip', icon='forward', button_style='warning')
next_button = widgets.Button(description='Next', icon='arrow-right')
label_output = widgets.Output()
state = {'position': 0, 'sample_ids': []}

def filtered_sample_ids() -> list[str]:
    subset = annotations_df[annotations_df['split'] == split_selector.value]
    filters = {'Pending': subset['status'] == 'PENDING', 'Labeled': subset['status'] == 'LABELED', 'Skipped': subset['status'] == 'SKIPPED'}
    if show_selector.value != 'All':
        subset = subset[filters[show_selector.value]]
    if split_selector.value == 'train' and show_selector.value == 'Pending' and ACTIVE_LEARNING_QUEUE_PATH.is_file():
        queue = pd.read_csv(ACTIVE_LEARNING_QUEUE_PATH).set_index('sample_id')
        subset = subset.assign(_priority=subset['sample_id'].map(queue['priority_score'])).sort_values('_priority', ascending=False, na_position='last')
    elif split_selector.value == 'train':
        subset = subset.sample(frac=1, random_state=42)
    return subset['sample_id'].tolist()

def persist_annotations() -> None:
    annotations_df.to_csv(ANNOTATION_PATH, index=False)

def render_current() -> None:
    state['sample_ids'] = filtered_sample_ids()
    if not state['sample_ids']:
        with label_output:
            clear_output(wait=True)
            print(f'No samples for {split_selector.value} / {show_selector.value}.')
        return
    state['position'] = min(max(state['position'], 0), len(state['sample_ids']) - 1)
    sample_id = state['sample_ids'][state['position']]
    row = annotations_df.loc[annotations_df['sample_id'] == sample_id].iloc[0]
    suggestion = ''
    if ACTIVE_LEARNING_QUEUE_PATH.is_file():
        queue = pd.read_csv(ACTIVE_LEARNING_QUEUE_PATH).fillna('')
        match = queue[queue['sample_id'] == sample_id]
        suggestion = str(match.iloc[0]['suggestion']) if not match.empty else ''
    plate_input.value = str(row['plate_text']) or suggestion
    notes_input.value = str(row['notes'])
    with label_output:
        clear_output(wait=True)
        labeled_count = int(((annotations_df['split'] == split_selector.value) & (annotations_df['status'] == 'LABELED')).sum())
        split_count = int((annotations_df['split'] == split_selector.value).sum())
        print(f'{state["position"] + 1}/{len(state["sample_ids"])} in view | labeled {labeled_count}/{split_count}')
        print(f'Sample: {sample_id}')
        print(f'Source reference (NOT ground truth): {row["reference_id"]}')
        print(f'Status: {row["status"]}')
        if suggestion:
            print(f'OCR suggestion (verify before saving): {suggestion}')
        crop = cv2.imread(str(row['crop_path']))
        if crop is not None:
            display(DisplayImage(filename=str(row['crop_path']), width=min(900, max(420, crop.shape[1] * 3))))
            print(f'Crop size: {crop.shape[1]}x{crop.shape[0]}')

def reset_view(_=None) -> None:
    state['position'] = 0
    render_current()

def move(step: int) -> None:
    state['position'] += step
    render_current()

def save_current(_=None) -> None:
    if not state['sample_ids']:
        return
    normalized = normalize_plate_text(plate_input.value)
    if not PLATE_PATTERN.fullmatch(normalized):
        with label_output:
            print(f"Invalid Indonesian plate format: {normalized or 'EMPTY'}")
        return
    sample_id = state['sample_ids'][state['position']]
    mask = annotations_df['sample_id'] == sample_id
    annotations_df.loc[mask, ['plate_text', 'status', 'notes']] = [normalized, 'LABELED', notes_input.value.strip()]
    persist_annotations()
    render_current()

def skip_current(_=None) -> None:
    if not state['sample_ids']:
        return
    sample_id = state['sample_ids'][state['position']]
    mask = annotations_df['sample_id'] == sample_id
    annotations_df.loc[mask, ['plate_text', 'status', 'notes']] = ['', 'SKIPPED', notes_input.value.strip() or 'Unreadable']
    persist_annotations()
    render_current()

split_selector.observe(reset_view, names='value')
show_selector.observe(reset_view, names='value')
previous_button.on_click(lambda _: move(-1))
next_button.on_click(lambda _: move(1))
save_button.on_click(save_current)
skip_button.on_click(skip_current)

controls = widgets.VBox([widgets.HBox([split_selector, show_selector]), plate_input, notes_input, widgets.HBox([previous_button, save_button, skip_button, next_button]), label_output])
display(controls)
render_current()


## 4. Labeling progress and quality checks

Jalankan cell ini kapan saja. Duplikasi crop lintas split dapat menyebabkan data leakage; hash pixel digunakan untuk mendeteksi file crop identik.

In [ ]:
annotations_df = pd.read_csv(ANNOTATION_PATH).fillna('')
progress_df = annotations_df.groupby(['split', 'status']).size().rename('samples').reset_index()
display(progress_df)

labeled_df = annotations_df[annotations_df['status'] == 'LABELED'].copy()
if not labeled_df.empty:
    print(f'Labeled: {len(labeled_df)}')
    print(f'Unique texts: {labeled_df["plate_text"].nunique()}')
    invalid_labels = labeled_df[~labeled_df['plate_text'].str.fullmatch(PLATE_PATTERN)]
    print(f'Invalid formats: {len(invalid_labels)}')
    display(labeled_df.groupby('split')['plate_text'].agg(samples='size', unique_texts='nunique'))


## 5. Export PaddleOCR recognition manifests

Format setiap baris adalah `relative/image/path\tLABEL`. Hanya status `LABELED` yang diekspor. Test tetap berada dalam manifest terpisah.

In [ ]:
EXPORT_ROOT = OUTPUT_ROOT / 'manifests'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
annotations_df = pd.read_csv(ANNOTATION_PATH).fillna('')

for split in SPLITS:
    split_df = annotations_df[(annotations_df['split'] == split) & (annotations_df['status'] == 'LABELED')].copy()
    manifest_path = EXPORT_ROOT / f'{split}.txt'
    lines = []
    for row in split_df.itertuples(index=False):
        relative_crop_path = Path(row.crop_path).relative_to(OUTPUT_ROOT).as_posix()
        lines.append(f'{relative_crop_path}\t{row.plate_text}')
    manifest_path.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
    print(f'{split}: {len(lines)} labels -> {manifest_path}')

print('Use train.txt for training, valid.txt for model selection, and test.txt only for final evaluation.')


## Labeling target

1. Label all readable `test` samples for a fixed benchmark.
2. Label all readable `valid` samples for model selection.
3. Label at least 500-1,000 diverse `train` samples, fine-tune, and inspect the learning curve before labeling the remaining train set.
4. Never infer ground truth from the source filename; `reference_id` is operational metadata, not the visible plate text.